In [1]:
!pip install jiwer openai-whisper speechbrain torchaudio librosa pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 127.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 63.4 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=836199c34a99d78f1765058491bb711ecfdd939b7babafbf1f0830cab3b39f40
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [3]:
import os
import glob
import pandas as pd
import whisper
import librosa
import numpy as np
import torch
from jiwer import wer
from speechbrain.inference.speaker import EncoderClassifier

# 1. Configuration
# The exact Hindi prompt you used to generate the audio
PROMPT_TEXT = "हैलो, मेरा नाम जय है।"
REFERENCE_AUDIO = "Demo-Audio.wav" # Only used for S-SIM scoring

# 2. Load Models
print("Loading Whisper ASR Model...")
asr_model = whisper.load_model("base")

print("Loading SpeechBrain Speaker Recognition Model...")
try:
    spk_model = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="tmp_dir"
    )
    has_spk_model = True
except Exception as e:
    print(f"Warning: Could not load SpeechBrain. S-SIM will be skipped. Error: {e}")
    has_spk_model = False

# 3. Helper Function for SNR
def calculate_snr(y):
    """Estimates Signal-to-Noise Ratio (SNR) in Decibels."""
    # Assume signal power is the mean of the squared amplitude
    signal_power = np.mean(y**2)
    # Estimate noise floor using the bottom 10% of amplitude (silences/background)
    noise_floor = np.mean(y[np.abs(y) < np.percentile(np.abs(y), 10)]**2) + 1e-10
    return 10 * np.log10(signal_power / noise_floor)

# 4. Evaluation Loop
print("\nStarting Evaluation Loop...")
results_list = []

# Find all .wav files in the current directory
audio_files = glob.glob("*.wav")

for audio_path in audio_files:
    # Skip the reference file itself so we don't evaluate it
    if audio_path == REFERENCE_AUDIO:
        continue

    print(f"Evaluating: {audio_path} ...")
    file_metrics = {"File Name": audio_path}

    try:
        # Load audio using librosa
        y, sr = librosa.load(audio_path, sr=16000)

        # --- Metric 1: Intelligibility (Word Error Rate) ---
        transcription = asr_model.transcribe(audio_path, language="hi")["text"]
        # Clean text for fair comparison (remove punctuation, lower case)
        clean_prompt = PROMPT_TEXT.replace("।", "").replace(",", "").strip()
        clean_trans = transcription.replace("।", "").replace(",", "").strip()

        file_metrics["Word Error Rate (WER)"] = round(wer(clean_prompt, clean_trans), 3)
        file_metrics["Transcribed Text"] = transcription.strip()

        # --- Metric 2: Clarity (Signal-to-Noise Ratio) ---
        file_metrics["SNR (dB)"] = round(calculate_snr(y), 2)

        # --- Metric 3: Speaker Similarity (Only if reference exists) ---
        if has_spk_model and os.path.exists(REFERENCE_AUDIO):
            sig_ref, _ = librosa.load(REFERENCE_AUDIO, sr=16000)

            # Convert to tensors
            emb_gen = spk_model.encode_batch(torch.tensor(y).unsqueeze(0))
            emb_ref = spk_model.encode_batch(torch.tensor(sig_ref).unsqueeze(0))

            # Calculate Cosine Similarity
            cos = torch.nn.CosineSimilarity(dim=-1)
            similarity = cos(emb_gen, emb_ref).item()
            file_metrics["Speaker Similarity (S-SIM)"] = round(similarity, 3)
        else:
            file_metrics["Speaker Similarity (S-SIM)"] = "N/A"

    except Exception as e:
        print(f"Error evaluating {audio_path}: {e}")
        file_metrics["Word Error Rate (WER)"] = "Error"
        file_metrics["SNR (dB)"] = "Error"
        file_metrics["Speaker Similarity (S-SIM)"] = "Error"
        file_metrics["Transcribed Text"] = "Error"

    results_list.append(file_metrics)

# 5. Build DataFrame and Save
df_results = pd.DataFrame(results_list)

# Sort alphabetically by file name
df_results = df_results.sort_values(by="File Name").reset_index(drop=True)

# Save to CSV
output_csv = "TTS_Objective_Metrics.csv"
df_results.to_csv(output_csv, index=False)

print(f"\n✅ Evaluation Complete! Results saved to {output_csv}")

# Display the beautiful dataframe right in Colab
import IPython
display(df_results)

Loading Whisper ASR Model...


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmp_dir/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmp_dir/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/tmp_dir/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmp_dir/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmp_dir/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Loading SpeechBrain Speaker Recognition Model...



Starting Evaluation Loop...
Evaluating: vits_rasa_male.wav ...
Evaluating: indic_parler_female.wav ...
Evaluating: indic_parler_male.wav ...
Evaluating: xtts_v2_male_hindi.wav ...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Evaluating: bark_hindi_female.wav ...
Evaluating: bark_hindi_male.wav ...
Evaluating: vits_rasa_female.wav ...
Evaluating: mms_male_hindi.wav ...

✅ Evaluation Complete! Results saved to TTS_Objective_Metrics.csv


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


,File Name,Word Error Rate (WER),Transcribed Text,SNR (dB),Speaker Similarity (S-SIM)
0,bark_hindi_female.wav,1.0,"Cheese Samnanha, Barikadhi",41.160000,-0.011
1,bark_hindi_male.wav,1.2,"Hi lo, My name is Jay.",50.599998,0.170
2,indic_parler_female.wav,1.0,"Hello, my name is Aay.",39.570000,0.166
3,indic_parler_male.wav,1.0,"Hello, Myranam Jayat",46.500000,0.035
4,mms_male_hindi.wav,1.0,"Namaskar, My Name is Jayay.",NaN,0.017
5,vits_rasa_female.wav,1.0,پیلو میرا نام جائے,75.239998,0.187
6,vits_rasa_male.wav,1.0,"Thailu, Myra Namjaya",74.570000,0.120
7,xtts_v2_male_hindi.wav,1.0,"Namaste, cheers !",NaN,-0.027


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [4]:
import os
import glob
import pandas as pd
import whisper
import librosa
import numpy as np
import torch
from jiwer import wer
from speechbrain.inference.speaker import EncoderClassifier

# 1. Configuration
PROMPT_TEXT = "हैलो, मेरा नाम जय है।"
REFERENCE_AUDIO = "Demo-Audio.wav" # Keep your reference as .wav or change to .mp3 if needed

# 2. Load Models
print("Loading Whisper ASR Model...")
asr_model = whisper.load_model("base")

print("Loading SpeechBrain Speaker Recognition Model...")
try:
    spk_model = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="tmp_dir"
    )
    has_spk_model = True
except Exception as e:
    print(f"Warning: Could not load SpeechBrain. S-SIM will be skipped. Error: {e}")
    has_spk_model = False

# 3. Helper Function for SNR
def calculate_snr(y):
    """Estimates Signal-to-Noise Ratio (SNR) in Decibels."""
    signal_power = np.mean(y**2)
    noise_floor = np.mean(y[np.abs(y) < np.percentile(np.abs(y), 10)]**2) + 1e-10
    return 10 * np.log10(signal_power / noise_floor)

# 4. Evaluation Loop
print("\nStarting Evaluation Loop for MP3 files...")
results_list = []

# 🚨 UPDATED: Now searching specifically for .mp3 files
audio_files = glob.glob("*.mp3")

if not audio_files:
    print("No .mp3 files found in the current directory!")

for audio_path in audio_files:
    # Skip the reference file if it happens to be an mp3
    if audio_path == REFERENCE_AUDIO:
        continue

    print(f"Evaluating: {audio_path} ...")
    file_metrics = {"File Name": audio_path}

    try:
        # Librosa natively decodes .mp3 files using ffmpeg
        y, sr = librosa.load(audio_path, sr=16000)

        # --- Metric 1: Intelligibility (Word Error Rate) ---
        # Whisper also natively reads .mp3
        transcription = asr_model.transcribe(audio_path, language="hi")["text"]

        clean_prompt = PROMPT_TEXT.replace("।", "").replace(",", "").strip()
        clean_trans = transcription.replace("।", "").replace(",", "").strip()

        file_metrics["Word Error Rate (WER)"] = round(wer(clean_prompt, clean_trans), 3)
        file_metrics["Transcribed Text"] = transcription.strip()

        # --- Metric 2: Clarity (Signal-to-Noise Ratio) ---
        file_metrics["SNR (dB)"] = round(calculate_snr(y), 2)

        # --- Metric 3: Speaker Similarity ---
        if has_spk_model and os.path.exists(REFERENCE_AUDIO):
            # We can compare an .mp3 generated file against a .wav reference safely!
            sig_ref, _ = librosa.load(REFERENCE_AUDIO, sr=16000)

            emb_gen = spk_model.encode_batch(torch.tensor(y).unsqueeze(0))
            emb_ref = spk_model.encode_batch(torch.tensor(sig_ref).unsqueeze(0))

            cos = torch.nn.CosineSimilarity(dim=-1)
            similarity = cos(emb_gen, emb_ref).item()
            file_metrics["Speaker Similarity (S-SIM)"] = round(similarity, 3)
        else:
            file_metrics["Speaker Similarity (S-SIM)"] = "N/A"

    except Exception as e:
        print(f"Error evaluating {audio_path}: {e}")
        file_metrics["Word Error Rate (WER)"] = "Error"
        file_metrics["SNR (dB)"] = "Error"
        file_metrics["Speaker Similarity (S-SIM)"] = "Error"
        file_metrics["Transcribed Text"] = "Error"

    results_list.append(file_metrics)

# 5. Build DataFrame and Save
if results_list:
    df_results = pd.DataFrame(results_list)
    df_results = df_results.sort_values(by="File Name").reset_index(drop=True)

    # Save to CSV
    output_csv = "TTSMaker_MP3_Metrics.csv"
    df_results.to_csv(output_csv, index=False)

    print(f"\n✅ Evaluation Complete! Results saved to {output_csv}")

    # Display the dataframe in Colab
    import IPython
    display(df_results)

Loading Whisper ASR Model...


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmp_dir/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmp_dir/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/tmp_dir/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmp_dir/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmp_dir/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Loading SpeechBrain Speaker Recognition Model...



Starting Evaluation Loop for MP3 files...
Evaluating: ttsmaker-male.mp3 ...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Evaluating: ttsmaker-female.mp3 ...

✅ Evaluation Complete! Results saved to TTSMaker_MP3_Metrics.csv


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


,File Name,Word Error Rate (WER),Transcribed Text,SNR (dB),Speaker Similarity (S-SIM)
0,ttsmaker-female.mp3,1.0,"Halo, My Name is Jai",NaN,0.024
1,ttsmaker-male.mp3,1.0,"Hello, My Name is Jai",NaN,0.160


In [6]:
import os
import glob
import re
import pandas as pd
import whisper
import librosa
import numpy as np
import torch
from jiwer import wer
from speechbrain.inference.speaker import EncoderClassifier

# ==========================================
# 1. CONFIGURATION
# ==========================================
# The exact Hindi prompt you used
PROMPT_TEXT = "हैलो, मेरा नाम जय है।"
# Your reference voice (can be .wav or .mp3)
REFERENCE_AUDIO = "Demo-Audio.wav"

# ==========================================
# 2. LOAD EVALUATION MODELS
# ==========================================
print("Loading Whisper ASR Model (Medium) for accurate Hindi transcription...")
# Upgraded to 'medium' to prevent hallucination on short audio
asr_model = whisper.load_model("medium")

print("Loading SpeechBrain for Speaker Similarity...")
try:
    spk_model = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-ecapa-voxceleb",
        savedir="tmp_dir"
    )
    has_spk_model = True
except Exception as e:
    print(f"Warning: Could not load SpeechBrain. Error: {e}")
    has_spk_model = False

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
def calculate_snr(y):
    """Estimates Signal-to-Noise Ratio (SNR) in Decibels."""
    signal_power = np.mean(y**2)
    noise_floor = np.mean(y[np.abs(y) < np.percentile(np.abs(y), 10)]**2) + 1e-10
    return 10 * np.log10(signal_power / noise_floor)

def clean_hindi_text(text):
    """Removes all English, punctuation, and zero-width characters, leaving only Devanagari."""
    return re.sub(r'[^\u0900-\u097F\s]', '', text).strip()

# ==========================================
# 4. MASTER EVALUATION LOOP
# ==========================================
print("\nStarting Evaluation Loop...")
results_list = []

# 🚨 Find BOTH .wav and .mp3 files
audio_files = glob.glob("*.wav") + glob.glob("*.mp3")

if not audio_files:
    print("No audio files found in the current directory!")

for audio_path in audio_files:
    # Skip evaluating the reference audio itself
    if audio_path == REFERENCE_AUDIO:
        continue

    print(f"Evaluating: {audio_path} ...")
    file_metrics = {"File Name": audio_path}

    try:
        # Librosa loads both .wav and .mp3 seamlessly
        y, sr = librosa.load(audio_path, sr=16000)

        # --- Metric 1: Intelligibility (Word Error Rate) ---
        # 🚨 task="transcribe" prevents Whisper from translating Hindi to English
        transcription_result = asr_model.transcribe(
            audio_path,
            language="hi",
            task="transcribe",
            fp16=False
        )
        transcription = transcription_result["text"]

        # Filter down to pure Hindi script
        clean_prompt = clean_hindi_text(PROMPT_TEXT)
        clean_trans = clean_hindi_text(transcription)

        # Calculate WER Safely
        if len(clean_prompt) > 0 and len(clean_trans) > 0:
            file_metrics["Word Error Rate (WER)"] = round(wer(clean_prompt, clean_trans), 3)
        else:
            # If the model output zero Devanagari, it failed the assignment
            file_metrics["Word Error Rate (WER)"] = 1.0

        file_metrics["Transcribed Text"] = transcription.strip()

        # --- Metric 2: Clarity (Signal-to-Noise Ratio) ---
        file_metrics["SNR (dB)"] = round(calculate_snr(y), 2)

        # --- Metric 3: Speaker Similarity (For voice cloning) ---
        if has_spk_model and os.path.exists(REFERENCE_AUDIO):
            sig_ref, _ = librosa.load(REFERENCE_AUDIO, sr=16000)

            emb_gen = spk_model.encode_batch(torch.tensor(y).unsqueeze(0))
            emb_ref = spk_model.encode_batch(torch.tensor(sig_ref).unsqueeze(0))

            cos = torch.nn.CosineSimilarity(dim=-1)
            similarity = cos(emb_gen, emb_ref).item()
            file_metrics["Speaker Similarity (S-SIM)"] = round(similarity, 3)
        else:
            file_metrics["Speaker Similarity (S-SIM)"] = "N/A"

    except Exception as e:
        print(f"Error evaluating {audio_path}: {e}")
        file_metrics["Word Error Rate (WER)"] = "Error"
        file_metrics["Transcribed Text"] = "Error"
        file_metrics["SNR (dB)"] = "Error"
        file_metrics["Speaker Similarity (S-SIM)"] = "Error"

    results_list.append(file_metrics)

# ==========================================
# 5. EXPORT RESULTS
# ==========================================
if results_list:
    df_results = pd.DataFrame(results_list)
    df_results = df_results.sort_values(by="File Name").reset_index(drop=True)

    # Save to CSV
    output_csv = "Master_TTS_Metrics.csv"
    df_results.to_csv(output_csv, index=False)

    print(f"\n✅ Evaluation Complete! Results saved to {output_csv}")

    # Display beautifully in Colab
    import IPython
    display(df_results)

Loading Whisper ASR Model (Medium) for accurate Hindi transcription...


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmp_dir/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmp_dir/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/tmp_dir/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmp_dir/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmp_dir/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Loading SpeechBrain for Speaker Similarity...



Starting Evaluation Loop...
Evaluating: vits_rasa_male.wav ...
Evaluating: indic_parler_female.wav ...
Evaluating: indic_parler_male.wav ...
Evaluating: xtts_v2_male_hindi.wav ...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


...

Evaluating: ttsmaker-male.mp3 ...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


Evaluating: ttsmaker-female.mp3 ...

✅ Evaluation Complete! Results saved to Master_TTS_Metrics.csv


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


,File Name,Word Error Rate (WER),Transcribed Text,SNR (dB),Speaker Similarity (S-SIM)
0,bark_hindi_female.wav,1.0,"चीस् सम्नन्हा, बरकादि!",41.160000,-0.011
1,bark_hindi_male.wav,0.8,"हाई लो, मेरा नाम जैहें",50.599998,0.170
2,indic_parler_female.wav,1.0,हाइ लोग भी नाम नहीं ये,39.570000,0.166
3,indic_parler_male.wav,1.0,"Hello, my name is Giant.",46.500000,0.035
4,mms_male_hindi.wav,0.6,नमस्कार मेरा नाम जैहे,NaN,0.017
5,ttsmaker-female.mp3,0.6,हालो! मेरा नाम जै है!,NaN,0.024
6,ttsmaker-male.mp3,1.0,"Hello, mera naam Jai hai.",NaN,0.160
7,vits_rasa_female.wav,0.6,"आईलो, मेरा नाम जैए",75.239998,0.187
8,vits_rasa_male.wav,0.6,"हेलो, मेरा नाम जैया",74.570000,0.120
9,xtts_v2_male_hindi.wav,0.6,नमस्ते! मेरा नाम पलक है!,NaN,-0.027
